# Elementos de Redes II
## Grafos estendidos, densidade, subgrafos, graus e matrizes

**Disciplina:** Estrutura de Dados II — Grafos  
**Formato:** aula prática e investigativa em Python  
**Biblioteca principal:** [NetworkX](https://networkx.org/)

> O tipo de grafo, a escala da rede e a forma de representação determinam quais perguntas podem ser respondidas — e quanto custará respondê-las.

## Objetivos de aprendizagem

Ao final desta aula, você deverá ser capaz de:

1. explicar o problema das Sete Pontes de Königsberg usando graus e caminhos eulerianos;
2. escolher entre `Graph`, `DiGraph`, `MultiGraph` e `MultiDiGraph`;
3. construir e interpretar grafos bipartidos, projeções e redes multilayer;
4. calcular densidade para grafos não direcionados, direcionados e bipartidos;
5. diferenciar rede densa de rede esparsa em termos de crescimento assintótico;
6. extrair subgrafos induzidos, subgrafos de arestas e redes egocentradas;
7. calcular grau médio e distribuição de graus;
8. interpretar matrizes de adjacência binárias, assimétricas, ponderadas e retangulares;
9. relacionar potências da matriz de adjacência a passeios e triângulos.

### Roteiro sugerido

| Bloco | Tema | Tempo sugerido |
|---|---|---:|
| 1 | Königsberg e grafos com arestas paralelas | 25 min |
| 2 | Tipos de grafos, bipartidos e multilayer | 35 min |
| 3 | Densidade, esparsidade e grau médio | 35 min |
| 4 | Subgrafos e ego networks | 25 min |
| 5 | Distribuição de graus e matrizes | 35 min |
| 6 | Exercícios e discussão | 30–45 min |


## 1. Configuração do ambiente

Todos os exemplos principais são autocontidos. Em vez de baixar grandes conjuntos de dados durante a aula, utilizaremos redes pequenas e a rede Karate, distribuída com o próprio NetworkX. Uma seção opcional ao final mostra como acessar os conjuntos SNAP do material de apoio.

In [ ]:
from collections import Counter
from itertools import combinations
from math import comb
import platform

import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import networkx as nx
import numpy as np
import pandas as pd
from IPython.display import display

SEED = 42
AZUL = "#2A6FBB"
LARANJA = "#E76F51"
VERDE = "#2A9D8F"
ROXO = "#7A5AF8"
CINZA = "#667085"

plt.rcParams.update({
    "figure.figsize": (8, 5),
    "axes.titlesize": 14,
    "axes.labelsize": 11,
})

print(f"Python: {platform.python_version()}")
print(f"NetworkX: {nx.__version__}")
print(f"NumPy: {np.__version__}")
print(f"pandas: {pd.__version__}")

## 2. O problema das Sete Pontes de Königsberg

Em 1735, Leonhard Euler estudou se seria possível atravessar cada uma das sete pontes de Königsberg exatamente uma vez. A forma das ilhas e margens não é relevante; importam apenas:

- cada porção de terra como um vértice;
- cada ponte como uma aresta;
- pontes diferentes entre o mesmo par de regiões como **arestas paralelas**.

Por isso, o modelo correto é um `MultiGraph`. Um `Graph` comum fundiria pontes paralelas e alteraria os graus, destruindo a estrutura do problema.

In [ ]:
K = nx.MultiGraph(name="Sete Pontes de Königsberg")

# A e B: margens; C e D: ilhas. Cada chave identifica uma ponte física.
pontes = [
    ("A", "C", "AC-1"),
    ("A", "C", "AC-2"),
    ("B", "C", "BC-1"),
    ("B", "C", "BC-2"),
    ("A", "D", "AD-1"),
    ("B", "D", "BD-1"),
    ("C", "D", "CD-1"),
]

for origem, destino, chave in pontes:
    K.add_edge(origem, destino, key=chave, ponte=chave)

print("Ordem |V|:", K.number_of_nodes())
print("Tamanho |E|:", K.number_of_edges())
print("Arestas com chaves:")
for u, v, chave in K.edges(keys=True):
    print(f"  {u} -- {v} | chave={chave}")

Em um multigrafo, a identidade de uma aresta é a tripla `(u, v, chave)`. A chave diferencia pontes paralelas entre as mesmas regiões.

In [ ]:
pos_K = {"A": (0, 1), "B": (0, -1), "C": (1.15, 0), "D": (2.3, 0)}

fig, ax = plt.subplots(figsize=(9, 5.5))
nx.draw_networkx_nodes(K, pos_K, node_color=AZUL, node_size=1800, ax=ax)
nx.draw_networkx_labels(K, pos_K, font_color="white", font_weight="bold", ax=ax)

# Curvaturas diferentes tornam as arestas paralelas visíveis.
for u, v in {tuple(sorted((u, v))) for u, v in K.edges()}:
    chaves = list(K[u][v].keys())
    curvaturas = np.linspace(-0.18, 0.18, len(chaves)) if len(chaves) > 1 else [0]
    for chave, curvatura in zip(chaves, curvaturas):
        nx.draw_networkx_edges(
            K,
            pos_K,
            edgelist=[(u, v, chave)],
            width=2.4,
            edge_color=CINZA,
            connectionstyle=f"arc3,rad={curvatura}",
            ax=ax,
        )

ax.set_title("Sete pontes: quatro regiões e sete arestas")
ax.axis("off")
plt.show()

## 3. Quatro combinações fundamentais

Direção e peso são dimensões independentes da modelagem.

| Tipo | Relação | Possível interpretação |
|---|---|---|
| não direcionado e não ponderado | $\{u,v\}$ | existência de amizade |
| não direcionado e ponderado | $\{u,v\}, w$ | distância de uma estrada |
| direcionado e não ponderado | $(u,v)$ | seguir alguém |
| direcionado e ponderado | $(u,v), w$ | tempo de um voo |

O peso deve ter significado e unidade documentados. Um peso pode representar custo, distância, capacidade, intensidade ou probabilidade; nem sempre “menor” significa “melhor”.

In [ ]:
G_uu = nx.Graph()
G_uu.add_edges_from([(1, 2), (1, 3), (2, 3), (3, 4)])

G_uw = nx.Graph()
G_uw.add_weighted_edges_from([(1, 2, 2.5), (1, 3, 1.2), (2, 3, 0.75), (3, 4, 1.8)])

G_du = nx.DiGraph()
G_du.add_edges_from([(1, 2), (2, 3), (3, 1), (3, 4)])

G_dw = nx.DiGraph()
G_dw.add_weighted_edges_from([(1, 2, 3), (2, 3, 2), (3, 1, 4), (3, 4, 1)])

grafos_tipo = [
    (G_uu, "Não direcionado, não ponderado", "#B9D9F2"),
    (G_uw, "Não direcionado, ponderado", "#A8DADC"),
    (G_du, "Direcionado, não ponderado", "#F4A261"),
    (G_dw, "Direcionado, ponderado", "#E9A6C7"),
]

pos_tipos = {1: (0, 1), 2: (1, 1), 3: (0.5, 0), 4: (0.5, -1)}
fig, axes = plt.subplots(2, 2, figsize=(11, 9))

for ax, (grafo, titulo, cor) in zip(axes.flat, grafos_tipo):
    opcoes_seta = {"arrowsize": 20} if grafo.is_directed() else {}
    nx.draw_networkx(
        grafo,
        pos_tipos,
        ax=ax,
        node_color=cor,
        node_size=1000,
        font_weight="bold",
        edge_color=CINZA,
        **opcoes_seta,
    )
    pesos = nx.get_edge_attributes(grafo, "weight")
    if pesos:
        nx.draw_networkx_edge_labels(grafo, pos_tipos, edge_labels=pesos, ax=ax)
    ax.set_title(titulo)
    ax.axis("off")

plt.tight_layout()
plt.savefig(
    "tipos_de_grafos.png",
    dpi=300,
    transparent=True,
    bbox_inches="tight",
    pad_inches=0.05
)
plt.show()

In [ ]:
resumo_tipos = []
for grafo, titulo, _ in grafos_tipo:
    resumo_tipos.append({
        "tipo": titulo,
        "classe": type(grafo).__name__,
        "direcionado": grafo.is_directed(),
        "multigrafo": grafo.is_multigraph(),
        "possui weight": bool(nx.get_edge_attributes(grafo, "weight")),
    })

display(pd.DataFrame(resumo_tipos))

## 4. Grafos bipartidos

Um grafo $G=(U,V,E)$ é bipartido quando os vértices podem ser separados em dois conjuntos disjuntos e toda aresta liga um vértice de $U$ a um vértice de $V$.

Exemplos: estudantes–disciplinas, autores–artigos, usuários–produtos e atores–filmes. Não há arestas estudante–estudante ou disciplina–disciplina no grafo bipartido original.

In [ ]:
B = nx.Graph(name="Matrículas")
alunos = ["Ana", "Bruno", "Carla", "Diego", "Eva"]
disciplinas = ["ED II", "Grafos", "IA", "Banco de Dados"]

B.add_nodes_from(alunos, bipartite=0, tipo="aluno")
B.add_nodes_from(disciplinas, bipartite=1, tipo="disciplina")
B.add_edges_from([
    ("Ana", "ED II"), ("Ana", "Grafos"),
    ("Bruno", "ED II"), ("Bruno", "Grafos"), ("Bruno", "IA"),
    ("Carla", "Grafos"), ("Carla", "IA"),
    ("Diego", "Banco de Dados"),
    ("Eva", "ED II"), ("Eva", "Banco de Dados"),
])

pos_B = nx.bipartite_layout(B, alunos, align="vertical")
fig, ax = plt.subplots(figsize=(10, 6))
nx.draw_networkx_nodes(B, pos_B, nodelist=alunos, node_color=AZUL, node_size=1500, ax=ax)
nx.draw_networkx_nodes(B, pos_B, nodelist=disciplinas, node_color=VERDE, node_size=2100, ax=ax)
nx.draw_networkx_edges(B, pos_B, edge_color="#98A2B3", width=1.8, ax=ax)
rotulos_B = {no: no for no in B.nodes}
rotulos_B["Banco de Dados"] = "Banco\nde Dados"
nx.draw_networkx_labels(
    B,
    pos_B,
    labels=rotulos_B,
    font_color="white",
    font_size=8,
    font_weight="bold",
    ax=ax,
)
ax.legend(handles=[
    Line2D([0], [0], marker="o", color="w", label="Alunos", markerfacecolor=AZUL, markersize=12),
    Line2D([0], [0], marker="o", color="w", label="Disciplinas", markerfacecolor=VERDE, markersize=12),
])
ax.set_title("Grafo bipartido de matrículas")
ax.axis("off")
plt.show()

print("É bipartido?", nx.is_bipartite(B))
print("Conjuntos obtidos pela coloração:", nx.bipartite.color(B))

### Matriz de biadjacência

Como as linhas representam $U$ e as colunas representam $V$, a matriz pode ser retangular. Sua dimensão é $|U|\times|V|$.

In [ ]:
biadj = np.array([
    [int(B.has_edge(aluno, disciplina)) for disciplina in disciplinas]
    for aluno in alunos
])

biadj_df = pd.DataFrame(biadj, index=alunos, columns=disciplinas)
display(biadj_df)
print("Dimensão:", biadj_df.shape)
print("Número de matrículas:", int(biadj.sum()))

### Projeções

Uma projeção sobre os alunos conecta duas pessoas quando elas compartilham ao menos uma disciplina. Na projeção ponderada, o peso é a quantidade de vizinhos compartilhados.

**Cuidado:** a projeção perde a identidade das disciplinas e pode criar muitas arestas a partir de uma única disciplina muito popular.

In [ ]:
P_alunos = nx.bipartite.weighted_projected_graph(B, alunos)
P_disciplinas = nx.bipartite.weighted_projected_graph(B, disciplinas)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, grafo, titulo, cor in [
    (axes[0], P_alunos, "Projeção nos alunos", AZUL),
    (axes[1], P_disciplinas, "Projeção nas disciplinas", VERDE),
]:
    pos = nx.spring_layout(grafo, seed=SEED)
    pesos = nx.get_edge_attributes(grafo, "weight")
    larguras = [1.2 + pesos[aresta] for aresta in grafo.edges]
    rotulos = {no: ("Banco\nde Dados" if no == "Banco de Dados" else no) for no in grafo.nodes}
    nx.draw_networkx(
        grafo,
        pos,
        ax=ax,
        labels=rotulos,
        node_color=cor,
        node_size=1900 if grafo is P_disciplinas else 1500,
        font_color="white",
        font_size=8,
        font_weight="bold",
        width=larguras,
        edge_color="#98A2B3",
    )
    nx.draw_networkx_edge_labels(grafo, pos, edge_labels=pesos, ax=ax)
    ax.set_title(titulo)
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
arestas_projecao = pd.DataFrame(
    [(u, v, dados["weight"]) for u, v, dados in P_alunos.edges(data=True)],
    columns=["aluno 1", "aluno 2", "disciplinas em comum"],
)
display(arestas_projecao.sort_values("disciplinas em comum", ascending=False))

## 5. Redes multilayer e multiplex

Uma rede multilayer possui mais de um tipo de relação, instante ou contexto. Em uma rede social, duas pessoas podem ser amigas e também colegas de trabalho.

Um `MultiGraph` com um atributo `layer` é uma codificação prática para casos simples. Modelos multilayer mais gerais também podem representar cópias de um mesmo ator em camadas diferentes e arestas entre camadas.

In [ ]:
M = nx.MultiGraph(name="Rede social multiplex")
M.add_nodes_from(["Alice", "Bob", "Carla", "Diana"])

relacoes = [
    ("Alice", "Bob", "amizade", 1),
    ("Alice", "Carla", "amizade", 1),
    ("Bob", "Carla", "amizade", 1),
    ("Alice", "Bob", "profissional", 2),  # aresta paralela em outra camada
    ("Alice", "Diana", "profissional", 2),
    ("Bob", "Diana", "profissional", 2),
    ("Carla", "Diana", "profissional", 2),
]

for indice, (u, v, camada, peso) in enumerate(relacoes):
    M.add_edge(u, v, key=f"{camada}-{indice}", layer=camada, weight=peso)

for u, v, chave, dados in M.edges(keys=True, data=True):
    print(f"{u} -- {v} | chave={chave} | {dados}")

In [ ]:
pos_M = nx.spring_layout(M, seed=SEED)
amizades = [(u, v, k) for u, v, k, d in M.edges(keys=True, data=True) if d["layer"] == "amizade"]
profissionais = [(u, v, k) for u, v, k, d in M.edges(keys=True, data=True) if d["layer"] == "profissional"]

fig, ax = plt.subplots(figsize=(8, 5.5))
nx.draw_networkx_nodes(M, pos_M, node_color="#DCEEFF", node_size=1600, ax=ax)
nx.draw_networkx_labels(M, pos_M, font_weight="bold", ax=ax)
nx.draw_networkx_edges(M, pos_M, edgelist=amizades, edge_color=AZUL, width=2.4, ax=ax)
nx.draw_networkx_edges(
    M,
    pos_M,
    edgelist=profissionais,
    edge_color=VERDE,
    width=2.4,
    style="dashed",
    connectionstyle="arc3,rad=0.12",
    ax=ax,
)
ax.legend(handles=[
    Line2D([0], [0], color=AZUL, lw=2.4, label="Amizade"),
    Line2D([0], [0], color=VERDE, lw=2.4, linestyle="dashed", label="Profissional"),
])
ax.set_title("Duas camadas de relacionamento")
ax.axis("off")
plt.show()

## 6. Densidade

A densidade é a fração de arestas possíveis que realmente existem.

### Grafo simples não direcionado, sem laços

$$L_{\max}=\binom{N}{2}=\frac{N(N-1)}{2}, \qquad
d=\frac{L}{L_{\max}}=\frac{2L}{N(N-1)}.$$

### Grafo simples direcionado, sem laços

$$L_{\max}=N(N-1), \qquad d=\frac{L}{N(N-1)}.$$

### Grafo bipartido simples

$$L_{\max}=|U||V|, \qquad d=\frac{L}{|U||V|}.$$

Essas fórmulas pressupõem grafos simples. Em multigrafos, o número de arestas paralelas não possui esse mesmo limite e `nx.density` pode produzir valor maior que 1; a interpretação deve ser redefinida para o domínio.

In [ ]:
def densidade_manual(G):
    if G.is_multigraph():
        raise nx.NetworkXError("A fórmula requer um grafo simples.")
    n = G.number_of_nodes()
    if n < 2:
        return 0.0
    m = G.number_of_edges()
    return m / (n * (n - 1)) if G.is_directed() else 2 * m / (n * (n - 1))


for grafo, nome in [
    (G_uu, "não direcionado"),
    (G_du, "direcionado"),
]:
    print(
        f"{nome:17s} | manual={densidade_manual(grafo):.3f} "
        f"| NetworkX={nx.density(grafo):.3f}"
    )

densidade_B = B.number_of_edges() / (len(alunos) * len(disciplinas))
print(f"bipartido         | manual={densidade_B:.3f}")
print(f"Königsberg (MultiGraph) | nx.density={nx.density(K):.3f} (> 1 é possível)")

### Densidade e grau médio

Em um grafo simples não direcionado:

$$\langle k\rangle=\frac{1}{N}\sum_i k_i=\frac{2L}{N}=d(N-1).$$

Logo, a densidade é a razão entre o grau médio observado e o grau máximo possível $N-1$:

$$d=\frac{\langle k\rangle}{N-1}.$$

In [ ]:
def resumo_nao_direcionado(G):
    n = G.number_of_nodes()
    m = G.number_of_edges()
    grau_medio = sum(dict(G.degree()).values()) / n if n else 0
    return {
        "N": n,
        "L": m,
        "densidade": nx.density(G),
        "grau médio": grau_medio,
        "d(N-1)": nx.density(G) * (n - 1) if n else 0,
    }


display(pd.DataFrame([resumo_nao_direcionado(G_uu)]))

## 7. Esparsidade e crescimento da rede

“Esparsa” não significa apenas “densidade pequena” em uma fotografia. É uma afirmação sobre como o número de arestas $L$ cresce quando o número de vértices $N$ aumenta.

- em uma família esparsa, tipicamente $L=O(N)$ ou cresce ainda mais lentamente;
- em uma família densa, $L=\Theta(N^2)$;
- uma rede pode ter muitas arestas em valor absoluto e ainda ser esparsa se $N$ for muito grande.

In [ ]:
tamanhos = np.array([10, 20, 40, 80, 160, 320])
arestas_ciclo = tamanhos                  # C_N possui N arestas
arestas_completo = tamanhos * (tamanhos - 1) // 2

crescimento_df = pd.DataFrame({
    "N": tamanhos,
    "L no ciclo": arestas_ciclo,
    "densidade do ciclo": 2 * arestas_ciclo / (tamanhos * (tamanhos - 1)),
    "L no completo": arestas_completo,
    "densidade do completo": np.ones_like(tamanhos, dtype=float),
})
display(crescimento_df)

plt.figure(figsize=(8, 5.5))
plt.loglog(tamanhos, arestas_ciclo, "o-", label=r"ciclo: $L=N$")
plt.loglog(tamanhos, arestas_completo, "s-", label=r"completo: $L=N(N-1)/2$")
plt.xlabel("número de vértices N")
plt.ylabel("número de arestas L")
plt.title("Crescimento linear versus quadrático")
plt.grid(alpha=0.25, which="both")
plt.legend()
plt.tight_layout()
plt.show()

## 8. Uma rede empírica autocontida: Karate Club

A rede do clube de karatê de Zachary representa relações sociais observadas entre 34 membros. Ela é pequena, mas permite praticar métricas, subgrafos e distribuição de graus sem downloads externos.

Os rótulos `Mr. Hi` e `Officer` indicam os dois grupos observados após uma divisão do clube. Eles são atributos dos vértices, não resultados de um algoritmo nesta aula.

In [ ]:
Z = nx.karate_club_graph()
graus_Z = dict(Z.degree())
cores_Z = [AZUL if Z.nodes[n]["club"] == "Mr. Hi" else LARANJA for n in Z.nodes]
tamanhos_Z = [250 + 75 * graus_Z[n] for n in Z.nodes]
pos_Z = nx.spring_layout(Z, seed=SEED)

fig, ax = plt.subplots(figsize=(9, 7))
nx.draw_networkx(
    Z,
    pos_Z,
    ax=ax,
    node_color=cores_Z,
    node_size=tamanhos_Z,
    font_color="white",
    font_size=8,
    edge_color="#B8C0CC",
    width=1.2,
)
ax.legend(handles=[
    Line2D([0], [0], marker="o", color="w", label="Mr. Hi", markerfacecolor=AZUL, markersize=10),
    Line2D([0], [0], marker="o", color="w", label="Officer", markerfacecolor=LARANJA, markersize=10),
])
ax.set_title("Karate Club: tamanho do nó proporcional ao grau")
ax.axis("off")
plt.show()

In [ ]:
resumo_Z = resumo_nao_direcionado(Z)
display(pd.DataFrame([resumo_Z]).round(4))

assert np.isclose(resumo_Z["grau médio"], 2 * Z.number_of_edges() / Z.number_of_nodes())
assert np.isclose(resumo_Z["grau médio"], resumo_Z["d(N-1)"])

## 9. Distribuição de graus

Se $n_k$ é o número de vértices de grau $k$, a distribuição empírica é:

$$P(k)=\frac{n_k}{N}.$$

A CCDF (função de distribuição acumulada complementar) mostra $P(K\ge k)$ e costuma ser mais estável que um histograma na cauda. Uma aparência aproximadamente linear em escala log-log **não é prova** de lei de potência; tamanho da amostra e comparação entre modelos importam.

In [ ]:
contagem_graus = Counter(graus_Z.values())
valores_k = np.array(sorted(contagem_graus))
frequencias = np.array([contagem_graus[k] for k in valores_k])
probabilidades = frequencias / Z.number_of_nodes()
ccdf = np.array([sum(g >= k for g in graus_Z.values()) / Z.number_of_nodes() for k in valores_k])

distribuicao_df = pd.DataFrame({
    "k": valores_k,
    "n_k": frequencias,
    "P(k)": probabilidades,
    "P(K >= k)": ccdf,
})
display(distribuicao_df.round(4))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))

axes[0].bar(valores_k, frequencias, color=AZUL, edgecolor="white")
axes[0].set_xlabel("grau k")
axes[0].set_ylabel("número de vértices $n_k$")
axes[0].set_title("Histograma de graus")

axes[1].loglog(valores_k, ccdf, "o-", color=LARANJA)
axes[1].set_xlabel("grau k (escala log)")
axes[1].set_ylabel(r"$P(K \geq k)$ (escala log)")
axes[1].set_title("CCDF empírica")
axes[1].grid(alpha=0.25, which="both")

plt.tight_layout()
plt.show()

## 10. Subgrafos

Há diferentes operações que recebem o nome “subgrafo”:

- **subgrafo induzido por vértices:** escolhemos $S\subseteq V$ e preservamos todas as arestas do original com as duas extremidades em $S$;
- **subgrafo induzido por arestas:** escolhemos um subconjunto de arestas e preservamos suas extremidades;
- **ego network:** subgrafo do vértice ego e dos vértices até uma distância definida.

`G.subgraph(S)` produz uma view estruturalmente congelada e vinculada ao grafo original. Use `.copy()` quando precisar de um objeto independente e modificável.

In [ ]:
vertices_escolhidos = [0, 1, 2, 3, 7, 8, 13]
Z_view = Z.subgraph(vertices_escolhidos)
Z_copia = Z_view.copy()

arestas_escolhidas = list(Z.edges(0))[:5]
Z_por_arestas = Z.edge_subgraph(arestas_escolhidas).copy()

print("Subgrafo por vértices:", Z_view.number_of_nodes(), "nós e", Z_view.number_of_edges(), "arestas")
print("A view é congelada?", nx.is_frozen(Z_view))
print("A cópia é congelada?", nx.is_frozen(Z_copia))
print("Subgrafo por arestas:", Z_por_arestas.number_of_nodes(), "nós e", Z_por_arestas.number_of_edges(), "arestas")

### Ego networks

In [ ]:
ego = max(Z, key=Z.degree)
ego_r1 = nx.ego_graph(Z, ego, radius=1)
ego_r2 = nx.ego_graph(Z, ego, radius=2)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, ego_grafo, raio in [(axes[0], ego_r1, 1), (axes[1], ego_r2, 2)]:
    cores = [LARANJA if no == ego else AZUL for no in ego_grafo]
    nx.draw_networkx(
        ego_grafo,
        {no: pos_Z[no] for no in ego_grafo},
        ax=ax,
        node_color=cores,
        node_size=600,
        font_color="white",
        font_size=8,
        edge_color="#B8C0CC",
    )
    ax.set_title(f"Ego {ego}, raio {raio}: {ego_grafo.number_of_nodes()} nós")
    ax.axis("off")

plt.tight_layout()
plt.show()

### Amostragem não é apenas redução de tamanho

Uma amostra induzida de vértices remove todas as arestas que chegam a vértices não selecionados. Graus, componentes e caminhos mudam. Portanto, métricas calculadas na amostra não devem ser tratadas automaticamente como estimativas sem viés da rede inteira.

In [ ]:
rng = np.random.default_rng(SEED)
rede_grande = nx.barabasi_albert_graph(500, 2, seed=SEED)
amostra_nos = rng.choice(list(rede_grande.nodes), size=100, replace=False).tolist()
rede_amostra = rede_grande.subgraph(amostra_nos).copy()

comparacao_amostra = pd.DataFrame([
    {"rede": "completa", **resumo_nao_direcionado(rede_grande), "componentes": nx.number_connected_components(rede_grande)},
    {"rede": "amostra induzida", **resumo_nao_direcionado(rede_amostra), "componentes": nx.number_connected_components(rede_amostra)},
])
display(comparacao_amostra.round(4))

## 11. Matrizes de adjacência

Dada uma ordenação dos vértices $(v_1,\ldots,v_N)$, a matriz de adjacência $A$ registra as relações entre pares.

- grafo simples não direcionado: $A$ é binária e simétrica;
- dígrafo: $A_{ij}$ representa $i\rightarrow j$ e $A$ pode ser assimétrica;
- grafo ponderado: $A_{ij}$ contém o peso, não apenas 0 ou 1;
- grafo bipartido: a biadjacência $|U|\times|V|$ pode ser retangular;
- multigrafo: a conversão padrão pode somar pesos de arestas paralelas, exigindo atenção à semântica.

A ordenação dos vértices faz parte da interpretação da matriz e deve ser preservada.

In [ ]:
R = nx.Graph()
R.add_edges_from([("A", "B"), ("A", "C"), ("B", "C"), ("C", "D")])
ordem_R = ["A", "B", "C", "D"]
A = nx.to_numpy_array(R, nodelist=ordem_R, dtype=int)

fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))
pos_R = {"A": (0, 1), "B": (1, 1), "C": (0.5, 0), "D": (0.5, -1)}
nx.draw_networkx(R, pos_R, ax=axes[0], node_color=AZUL, node_size=900, font_color="white")
axes[0].set_title("Grafo")
axes[0].axis("off")

imagem = axes[1].imshow(A, cmap="Blues", vmin=0, vmax=1)
axes[1].set_xticks(range(len(ordem_R)), ordem_R)
axes[1].set_yticks(range(len(ordem_R)), ordem_R)
for i in range(len(ordem_R)):
    for j in range(len(ordem_R)):
        axes[1].text(j, i, A[i, j], ha="center", va="center")
axes[1].set_title("Matriz de adjacência")

plt.tight_layout()
plt.show()

display(pd.DataFrame(A, index=ordem_R, columns=ordem_R))
print("Simétrica?", np.array_equal(A, A.T))

In [ ]:
D_mat = nx.DiGraph()
D_mat.add_edges_from([("A", "B"), ("B", "C"), ("C", "A"), ("C", "D")])

W_mat = nx.Graph()
W_mat.add_weighted_edges_from([("A", "B", 2.5), ("A", "C", 1.2), ("B", "C", 0.75), ("C", "D", 1.8)])

A_dir = nx.to_numpy_array(D_mat, nodelist=ordem_R, dtype=int)
A_peso = nx.to_numpy_array(W_mat, nodelist=ordem_R, weight="weight")

fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))
for ax, matriz, titulo, mapa in [
    (axes[0], A_dir, "Direcionada: matriz assimétrica", "Oranges"),
    (axes[1], A_peso, "Ponderada: matriz não binária", "Purples"),
]:
    ax.imshow(matriz, cmap=mapa)
    ax.set_xticks(range(len(ordem_R)), ordem_R)
    ax.set_yticks(range(len(ordem_R)), ordem_R)
    for i in range(len(ordem_R)):
        for j in range(len(ordem_R)):
            ax.text(j, i, f"{matriz[i, j]:g}", ha="center", va="center")
    ax.set_title(titulo)

plt.tight_layout()
plt.show()

### Custo de uma matriz densa

Uma matriz densa possui $N^2$ posições, inclusive para pares sem aresta. Em redes esparsas grandes, isso pode ser proibitivo. Listas/dicionários de adjacência e matrizes esparsas armazenam principalmente as relações existentes.

In [ ]:
def tamanho_legivel(numero_bytes):
    unidades = ["B", "KiB", "MiB", "GiB", "TiB"]
    valor = float(numero_bytes)
    for unidade in unidades:
        if valor < 1024 or unidade == unidades[-1]:
            return f"{valor:.2f} {unidade}"
        valor /= 1024


memoria_df = pd.DataFrame({
    "N": [1_000, 10_000, 100_000],
})
memoria_df["entradas N²"] = memoria_df["N"] ** 2
memoria_df["float64 denso"] = memoria_df["entradas N²"].map(lambda x: tamanho_legivel(8 * x))
display(memoria_df)

## 12. Conjuntos de dados externos — seção opcional

Os slides utilizam conjuntos da coleção [SNAP](https://snap.stanford.edu/data/): Enron Email, Patent Citation e Twitch Gamers. Eles são úteis, mas alguns possuem milhões de relações e não devem ser downloads obrigatórios de uma aula introdutória.

O carregador abaixo processa um arquivo `.gz` em fluxo e permite impor um limite de arestas. A chamada de rede fica **desativada por padrão**. Antes de usar um conjunto externo, confira tamanho, licença, direção das arestas, presença de duplicatas e significado dos identificadores.

In [ ]:
import gzip
import urllib.request


def carregar_edgelist_gz(url, *, direcionado=False, limite_arestas=None):
    G = nx.DiGraph() if direcionado else nx.Graph()
    with urllib.request.urlopen(url) as resposta:
        with gzip.GzipFile(fileobj=resposta) as arquivo:
            for linha_bytes in arquivo:
                linha = linha_bytes.decode("utf-8").strip()
                if not linha or linha.startswith("#"):
                    continue
                origem, destino, *_ = linha.split()
                G.add_edge(int(origem), int(destino))
                if limite_arestas is not None and G.number_of_edges() >= limite_arestas:
                    break
    return G


EXECUTAR_DOWNLOAD_EXTERNO = True
if EXECUTAR_DOWNLOAD_EXTERNO:
    enron = carregar_edgelist_gz(
        "https://snap.stanford.edu/data/email-Enron.txt.gz",
        direcionado=False,
        limite_arestas=50_000,
    )
    print(resumo_nao_direcionado(enron))
else:
    print("Download externo desativado. Altere a variável conscientemente para executar.")

## 13. Prática guiada

### Exercício 1 — diagnóstico euleriano

Implemente `diagnostico_euleriano(G)` retornando um dicionário com os vértices de grau ímpar, a existência de trilha euleriana e a existência de circuito euleriano.

In [ ]:
def diagnostico_euleriano(G):
    impares = sorted((no for no, grau in G.degree() if grau % 2), key=str)
    return {
        "ímpares": impares,
        "possui_trilha": nx.has_eulerian_path(G),
        "possui_circuito": nx.is_eulerian(G),
    }

In [ ]:
assert diagnostico_euleriano(K)["ímpares"] == ["A", "B", "C", "D"]
assert not diagnostico_euleriano(K)["possui_trilha"]

ciclo_5 = nx.cycle_graph(5)
assert diagnostico_euleriano(ciclo_5)["possui_circuito"]
print("Teste do Exercício 1 aprovado.")

### Exercício 2 — densidade bipartida

Implemente `densidade_bipartida_manual(G, U, V)`. Valide se toda aresta cruza as partições e trate partições vazias.

In [ ]:
def densidade_bipartida_manual(G, U, V):
    U, V = set(U), set(V)
    if U & V:
        raise ValueError("As partições devem ser disjuntas.")
    if not U or not V:
        return 0.0
    if any(not ((u in U and v in V) or (u in V and v in U)) for u, v in G.edges()):
        raise ValueError("Existe uma aresta dentro de uma partição.")
    return G.number_of_edges() / (len(U) * len(V))

In [ ]:
assert np.isclose(densidade_bipartida_manual(B, alunos, disciplinas), 0.5)
print("Teste do Exercício 2 aprovado.")

### Exercício 3 — projeção explicável

Implemente uma função que retorne, para cada par de alunos conectado na projeção, o conjunto de disciplinas compartilhadas. Não use apenas o peso: preserve a explicação da aresta projetada.

In [ ]:
def explicar_projecao(G_bipartido, conjunto_alvo):
    explicacoes = {}
    for u, v in combinations(conjunto_alvo, 2):
        comuns = set(G_bipartido.neighbors(u)) & set(G_bipartido.neighbors(v))
        if comuns:
            explicacoes[(u, v)] = comuns
    return explicacoes

In [ ]:
explicacoes = explicar_projecao(B, alunos)
assert explicacoes[("Ana", "Bruno")] == {"ED II", "Grafos"}
assert ("Ana", "Diego") not in explicacoes
print("Teste do Exercício 3 aprovado.")

### Exercício 4 — resumo de uma ego network

Implemente `resumo_ego(G, ego, raio=1)` retornando número de nós, número de arestas, densidade e vizinhos diretos do ego.

In [ ]:
def resumo_ego(G, ego, raio=1):
    E = nx.ego_graph(G, ego, radius=raio)
    return {
        "nós": E.number_of_nodes(),
        "arestas": E.number_of_edges(),
        "densidade": nx.density(E),
        "vizinhos diretos": set(G.neighbors(ego)),
    }

In [ ]:
resumo_0 = resumo_ego(Z, 0)
assert resumo_0["nós"] == Z.degree(0) + 1
assert 0 <= resumo_0["densidade"] <= 1
print("Teste do Exercício 4 aprovado.")

### Exercício 5 — triângulos pela matriz

Implemente `triangulos_por_matriz(G, ordem=None)` para grafos simples não direcionados e compare o resultado com `nx.triangles`.

In [ ]:
def triangulos_por_matriz(G, ordem=None):
    if G.is_directed() or G.is_multigraph():
        raise nx.NetworkXError("Use um grafo simples não direcionado.")
    ordem = list(G.nodes) if ordem is None else list(ordem)
    matriz = nx.to_numpy_array(G, nodelist=ordem, weight=None)
    return int(round(np.trace(matriz @ matriz @ matriz) / 6))

In [ ]:
assert triangulos_por_matriz(R, ordem_R) == 1
assert triangulos_por_matriz(Z) == sum(nx.triangles(Z).values()) // 3
print("Teste do Exercício 5 aprovado.")

## 14. Checklist de modelagem e análise

- [ ] A estrutura aceita direção, pesos, laços e paralelismo conforme o problema?
- [ ] Cada aresta física ou evento foi preservado, ou houve agregação documentada?
- [ ] A fórmula de densidade corresponde ao tipo de grafo?
- [ ] “Esparsa” está sendo usada como propriedade de escala, não apenas como adjetivo visual?
- [ ] O subgrafo é induzido por vértices, por arestas ou por distância ao ego?
- [ ] A amostragem alterou graus, componentes ou alcançabilidade?
- [ ] A distribuição de graus foi normalizada por $N$?
- [ ] A ordem dos vértices associada à matriz foi preservada?
- [ ] Uma matriz densa cabe na memória para o valor de $N$ considerado?
- [ ] Resultados em projeções podem ser explicados pelos vizinhos bipartidos originais?

## 15. Síntese e próximas etapas

Nesta aula, usamos o problema de Königsberg para mostrar que a estrutura correta importa: um `Graph` eliminaria pontes paralelas e invalidaria a análise. Em seguida, examinamos grafos ponderados e direcionados, redes bipartidas e multilayer, fórmulas de densidade, crescimento esparso e denso, subgrafos, ego networks, distribuição de graus e representações matriciais.

A álgebra de matrizes revelou que representações não servem apenas para armazenar dados: elas também permitem contar passeios e estruturas locais. Ao mesmo tempo, o custo $O(N^2)$ de uma matriz densa reforça a importância de representações esparsas.

### Leituras e prática

- [Documentação oficial do NetworkX](https://networkx.org/documentation/stable/)
- [Tutorial oficial do NetworkX](https://networkx.org/documentation/stable/tutorial.html)
- [NetworkX — algoritmos bipartidos](https://networkx.org/documentation/stable/reference/algorithms/bipartite.html)
- [SNAP datasets](https://snap.stanford.edu/data/)
- Michele Coscia, *The Atlas for the Aspiring Network Scientist* — [networkatlas.eu](https://www.networkatlas.eu/)
